# 🎙️ Voice Studio — Free NVIDIA Cloud GPU Server
### Run State-of-the-Art F5-TTS Zero-Shot Voice Cloning on Free Google Colab NVIDIA T4 GPU

**Features:**
- ⚡ **Zero Quota Limits**: Unlimited voice cloning runs with 16GB VRAM.
- 🚀 **Blazing Fast**: Generates full voiceover paragraphs in ~1–2 seconds.
- 🔗 **1-Click Connect**: Gives you a public `https://....gradio.live` link to paste into your local Voice Studio app.

In [ ]:
# Step 1: Install High-Performance Voice Cloning Dependencies
!pip install -q f5-tts gradio soundfile torch torchaudio

In [ ]:
# Step 2: Initialize F5-TTS on Free NVIDIA GPU
import torch
import soundfile as sf
import gradio as gr
from f5_tts.api import F5TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Running on Device: {device.upper()}")
if device == "cuda":
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

f5 = F5TTS(model_type="F5-TTS", device=device)
print("✅ F5-TTS Model loaded successfully into GPU VRAM!")

In [ ]:
# Step 3: Launch Public Cloud API Server
def clone_voice(ref_audio, ref_text, gen_text, remove_silence=False):
    if not ref_audio:
        raise gr.Error("Reference audio is required!")
    if not gen_text:
        raise gr.Error("Generation text is required!")
        
    wav, sr, _ = f5.infer(
        ref_file=ref_audio,
        ref_text=ref_text or "",
        gen_text=gen_text,
        remove_silence=remove_silence
    )
    out_path = "/tmp/voice_studio_output.wav"
    sf.write(out_path, wav, sr)
    return out_path

api = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Audio(type="filepath", label="Reference Voice Sample"),
        gr.Textbox(label="Reference Transcript (Optional)"),
        gr.Textbox(label="Target Speech Text"),
        gr.Checkbox(label="Remove Silence", value=False)
    ],
    outputs=gr.Audio(type="filepath", label="Cloned Speech Output"),
    title="Voice Studio Cloud GPU Server",
    description="Dedicated NVIDIA T4 GPU endpoint for your local Voice Studio application."
)

# Launch with public share link
api.launch(share=True)